In [2]:
# =============================================================================
# 07_download_sentinel2.ipynb
#
# Purpose:
# Download cloud-masked Sentinel-2 Level-2A image chips for:
#
#   1. Treatment sites before Hurricane Helene
#   2. Treatment sites after Hurricane Helene
#   3. Counterfactual sites before Hurricane Helene
#   4. Counterfactual sites after Hurricane Helene
#
# Inputs:
#
# datasets/finals/treatment_sites.geojson
# datasets/finals/counterfactual_sites.geojson
#
# All Sentinel-2 outputs:
#
# datasets/finals/sentinel2/
# ├── treatment/
# │   ├── before/
# │   └── after/
# ├── counterfactual/
# │   ├── before/
# │   └── after/
# ├── previews/
# ├── sentinel2_image_inventory.csv
# ├── sentinel2_image_validation.csv
# ├── sentinel2_site_completeness.csv
# ├── sentinel2_matched_pair_completeness.csv
# ├── sentinel2_analysis_ready_inventory.csv
# └── sentinel2_folder_summary.csv
#
# Exported GeoTIFF band order:
#
# Band 1: B2   — Blue
# Band 2: B3   — Green
# Band 3: B4   — Red
# Band 4: B8   — Near infrared
# Band 5: B11  — Shortwave infrared 1
# Band 6: B12  — Shortwave infrared 2
# Band 7: NDVI
# Band 8: NDWI
# =============================================================================


# =============================================================================
# 1. Load packages
# =============================================================================

from pathlib import Path
import json
import time

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from tqdm.auto import tqdm

print("Packages loaded successfully.")


# =============================================================================
# 2. Define project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)

FINALS_DIR = BASE_DIR / "finals"

# Shared site files created by Notebook 06
TREATMENT_SITES_FILE = (
    FINALS_DIR /
    "treatment_sites.geojson"
)

COUNTERFACTUAL_SITES_FILE = (
    FINALS_DIR /
    "counterfactual_sites.geojson"
)

# Sentinel-2 root folder
SENTINEL2_DIR = (
    FINALS_DIR /
    "sentinel2"
)

S2_TREATMENT_DIR = (
    SENTINEL2_DIR /
    "treatment"
)

S2_COUNTERFACTUAL_DIR = (
    SENTINEL2_DIR /
    "counterfactual"
)

S2_TREATMENT_BEFORE_DIR = (
    S2_TREATMENT_DIR /
    "before"
)

S2_TREATMENT_AFTER_DIR = (
    S2_TREATMENT_DIR /
    "after"
)

S2_COUNTERFACTUAL_BEFORE_DIR = (
    S2_COUNTERFACTUAL_DIR /
    "before"
)

S2_COUNTERFACTUAL_AFTER_DIR = (
    S2_COUNTERFACTUAL_DIR /
    "after"
)

S2_PREVIEW_DIR = (
    SENTINEL2_DIR /
    "previews"
)

for folder in [
    FINALS_DIR,
    SENTINEL2_DIR,
    S2_TREATMENT_DIR,
    S2_COUNTERFACTUAL_DIR,
    S2_TREATMENT_BEFORE_DIR,
    S2_TREATMENT_AFTER_DIR,
    S2_COUNTERFACTUAL_BEFORE_DIR,
    S2_COUNTERFACTUAL_AFTER_DIR,
    S2_PREVIEW_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Sentinel-2 output directory:")
print(SENTINEL2_DIR)


# =============================================================================
# 3. Check required input files
# =============================================================================

required_files = [
    TREATMENT_SITES_FILE,
    COUNTERFACTUAL_SITES_FILE,
]

missing_files = [
    file
    for file in required_files
    if not file.exists()
]

if missing_files:

    raise FileNotFoundError(
        "One or more site files are missing.\n"
        "Run 06_create_sites.ipynb first.\n\n"
        "Missing files:\n"
        + "\n".join(
            str(file)
            for file in missing_files
        )
    )

print("Required site files found.")


# =============================================================================
# 4. Authenticate and initialize Google Earth Engine
# =============================================================================

EARTH_ENGINE_PROJECT = "hurricane-504721"

try:

    ee.Initialize(
        project=EARTH_ENGINE_PROJECT
    )

    print(
        "Earth Engine initialized using "
        "existing credentials."
    )

except Exception as initialization_error:

    print(
        "Earth Engine initialization failed."
    )

    print(
        initialization_error
    )

    print(
        "\nStarting Earth Engine authentication..."
    )

    ee.Authenticate()

    ee.Initialize(
        project=EARTH_ENGINE_PROJECT
    )

    print(
        "Earth Engine authenticated and initialized."
    )


# =============================================================================
# 5. Define before and after periods
# =============================================================================

IMAGE_PERIODS = {
    "before": {
        "start": "2024-08-15",
        "end": "2024-09-23",
    },
    "after": {
        "start": "2024-10-01",
        "end": "2024-10-31",
    },
}

print("Image periods:")

for period, dates in IMAGE_PERIODS.items():

    print(
        period,
        dates["start"],
        dates["end"],
    )


# =============================================================================
# 6. Define Sentinel-2 settings
# =============================================================================

SENTINEL2_COLLECTION = (
    "COPERNICUS/S2_SR_HARMONIZED"
)

MAX_SCENE_CLOUD_PERCENT = 80

EXPORT_SCALE_METERS = 10

EXPORT_CRS = "EPSG:32617"

OVERWRITE_EXISTING = False

CREATE_RGB_PREVIEWS = True

TEST_DOWNLOAD_FIRST = True

RETRY_ATTEMPTS = 3

RETRY_WAIT_SECONDS = 10

BAND_NAMES = [
    "B2",
    "B3",
    "B4",
    "B8",
    "B11",
    "B12",
    "NDVI",
    "NDWI",
]


# =============================================================================
# 7. Load treatment and counterfactual site polygons
# =============================================================================

treatment_sites = gpd.read_file(
    TREATMENT_SITES_FILE
)

counterfactual_sites = gpd.read_file(
    COUNTERFACTUAL_SITES_FILE
)


def standardize_site_layer(
    site_layer,
    expected_group,
):
    """
    Standardize one treatment or counterfactual site layer.
    """

    output = site_layer.copy()

    if output.crs is None:

        output = output.set_crs(
            "EPSG:4326"
        )

    else:

        output = output.to_crs(
            "EPSG:4326"
        )

    required_columns = [
        "site_id",
        "pair_id",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in output.columns
    ]

    if missing_columns:

        raise ValueError(
            f"The {expected_group} site file is missing "
            f"required columns: {missing_columns}"
        )

    output["group"] = expected_group

    output = output.loc[
        output.geometry.notna()
    ].copy()

    output = output.loc[
        ~output.geometry.is_empty
    ].copy()

    output = output.reset_index(
        drop=True
    )

    return output


treatment_sites = standardize_site_layer(
    treatment_sites,
    "treatment",
)

counterfactual_sites = standardize_site_layer(
    counterfactual_sites,
    "counterfactual",
)

print(
    "Treatment sites:",
    len(treatment_sites),
)

print(
    "Counterfactual sites:",
    len(counterfactual_sites),
)


# =============================================================================
# 8. Verify treatment and counterfactual pair alignment
# =============================================================================

treatment_pairs = set(
    treatment_sites["pair_id"]
)

counterfactual_pairs = set(
    counterfactual_sites["pair_id"]
)

missing_counterfactual_pairs = (
    treatment_pairs -
    counterfactual_pairs
)

missing_treatment_pairs = (
    counterfactual_pairs -
    treatment_pairs
)

if missing_counterfactual_pairs:

    print(
        "Warning: treatment pairs without "
        "counterfactual sites:"
    )

    print(
        sorted(
            missing_counterfactual_pairs
        )
    )

if missing_treatment_pairs:

    print(
        "Warning: counterfactual pairs without "
        "treatment sites:"
    )

    print(
        sorted(
            missing_treatment_pairs
        )
    )


# =============================================================================
# 9. Combine all sites
# =============================================================================

all_sites = gpd.GeoDataFrame(
    pd.concat(
        [
            treatment_sites,
            counterfactual_sites,
        ],
        ignore_index=True,
    ),
    crs="EPSG:4326",
)

all_sites = (
    all_sites
    .sort_values(
        [
            "pair_id",
            "group",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Total sites:",
    len(all_sites),
)


# =============================================================================
# 10. Convert Shapely geometry to Earth Engine geometry
# =============================================================================

def shapely_to_ee_geometry(
    geometry,
):
    """
    Convert one Shapely geometry to an Earth Engine geometry.
    """

    geometry_json = json.loads(
        gpd.GeoSeries(
            [geometry],
            crs="EPSG:4326",
        ).to_json()
    )

    feature = (
        geometry_json[
            "features"
        ][0]
    )

    return ee.Geometry(
        feature[
            "geometry"
        ]
    )


# =============================================================================
# 11. Define Sentinel-2 cloud and shadow mask
#
# SCL classes removed:
#
# 0  = No data
# 1  = Saturated or defective
# 3  = Cloud shadow
# 8  = Medium-probability cloud
# 9  = High-probability cloud
# 10 = Thin cirrus
# 11 = Snow or ice
# =============================================================================

def mask_sentinel2_clouds(
    image,
):
    """
    Mask cloudy and invalid pixels and convert surface reflectance
    values to approximately 0–1.
    """

    scl = image.select(
        "SCL"
    )

    clear_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    reflectance = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8",
                "B11",
                "B12",
            ]
        )
        .multiply(
            0.0001
        )
        .toFloat()
    )

    return (
        reflectance
        .updateMask(
            clear_mask
        )
        .copyProperties(
            image,
            image.propertyNames(),
        )
    )


# =============================================================================
# 12. Diagnostic Sentinel-2 band check
# =============================================================================

diagnostic_collection = (
    ee.ImageCollection(
        SENTINEL2_COLLECTION
    )
    .filterDate(
        IMAGE_PERIODS["before"]["start"],
        IMAGE_PERIODS["before"]["end"],
    )
)

diagnostic_image = ee.Image(
    diagnostic_collection.first()
)

print("\nAvailable Sentinel-2 bands:")

print(
    diagnostic_image
    .bandNames()
    .getInfo()
)


# =============================================================================
# 13. Build Sentinel-2 median composite
# =============================================================================

def build_sentinel2_composite(
    region,
    start_date,
    end_date,
):
    """
    Create a cloud-masked median composite with reflectance,
    NDVI, and NDWI bands.
    """

    collection = (
        ee.ImageCollection(
            SENTINEL2_COLLECTION
        )
        .filterBounds(
            region
        )
        .filterDate(
            start_date,
            end_date,
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                MAX_SCENE_CLOUD_PERCENT,
            )
        )
    )

    scene_count = (
        collection
        .size()
        .getInfo()
    )

    if scene_count == 0:

        return None, 0

    masked_collection = (
        collection.map(
            mask_sentinel2_clouds
        )
    )

    composite = (
        masked_collection
        .median()
        .clip(
            region
        )
    )

    ndvi = (
        composite
        .normalizedDifference(
            [
                "B8",
                "B4",
            ]
        )
        .rename(
            "NDVI"
        )
    )

    ndwi = (
        composite
        .normalizedDifference(
            [
                "B3",
                "B8",
            ]
        )
        .rename(
            "NDWI"
        )
    )

    final_image = (
        composite
        .addBands(
            ndvi
        )
        .addBands(
            ndwi
        )
        .select(
            BAND_NAMES
        )
        .toFloat()
    )

    return (
        final_image,
        scene_count,
    )


# =============================================================================
# 14. Determine output directory
# =============================================================================

def get_output_directory(
    group,
    period,
):
    """
    Return the correct Sentinel-2 output folder.
    """

    folder_map = {
        (
            "treatment",
            "before",
        ): S2_TREATMENT_BEFORE_DIR,

        (
            "treatment",
            "after",
        ): S2_TREATMENT_AFTER_DIR,

        (
            "counterfactual",
            "before",
        ): S2_COUNTERFACTUAL_BEFORE_DIR,

        (
            "counterfactual",
            "after",
        ): S2_COUNTERFACTUAL_AFTER_DIR,
    }

    key = (
        group,
        period,
    )

    if key not in folder_map:

        raise ValueError(
            f"Unsupported group-period combination: {key}"
        )

    return folder_map[
        key
    ]


# =============================================================================
# 15. Create RGB preview
#
# GeoTIFF band order:
#
# 1 = B2, blue
# 2 = B3, green
# 3 = B4, red
# =============================================================================

def create_rgb_preview(
    tiff_file,
    preview_file,
):
    """
    Create a percentile-stretched natural-color PNG preview.
    """

    tiff_file = Path(
        tiff_file
    )

    preview_file = Path(
        preview_file
    )

    with rasterio.open(
        tiff_file
    ) as source:

        blue = source.read(
            1,
            masked=True,
        ).filled(
            np.nan
        )

        green = source.read(
            2,
            masked=True,
        ).filled(
            np.nan
        )

        red = source.read(
            3,
            masked=True,
        ).filled(
            np.nan
        )

    rgb = np.stack(
        [
            red,
            green,
            blue,
        ],
        axis=-1,
    )

    valid_values = rgb[
        np.isfinite(
            rgb
        )
    ]

    if valid_values.size == 0:

        print(
            f"No valid RGB pixels for {tiff_file.name}"
        )

        return False

    lower = np.nanpercentile(
        valid_values,
        2,
    )

    upper = np.nanpercentile(
        valid_values,
        98,
    )

    if upper <= lower:

        upper = lower + 1e-6

    stretched = np.clip(
        (
            rgb -
            lower
        )
        /
        (
            upper -
            lower
        ),
        0,
        1,
    )

    stretched[
        ~np.isfinite(
            stretched
        )
    ] = 0

    preview_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.figure(
        figsize=(6, 6)
    )

    plt.imshow(
        stretched
    )

    plt.axis(
        "off"
    )

    plt.tight_layout(
        pad=0
    )

    plt.savefig(
        preview_file,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close()

    return True


# =============================================================================
# 16. Download one Sentinel-2 site-period image
# =============================================================================

def download_site_image(
    site_row,
    period,
):
    """
    Build and download one Sentinel-2 image chip.
    """

    site_id = str(
        site_row["site_id"]
    )

    pair_id = str(
        site_row["pair_id"]
    )

    group = str(
        site_row["group"]
    )

    start_date = (
        IMAGE_PERIODS[
            period
        ]["start"]
    )

    end_date = (
        IMAGE_PERIODS[
            period
        ]["end"]
    )

    output_directory = get_output_directory(
        group,
        period,
    )

    output_file = (
        output_directory /
        f"{site_id}_{period}_sentinel2.tif"
    )

    preview_file = (
        S2_PREVIEW_DIR /
        group /
        period /
        f"{site_id}_{period}_rgb.png"
    )

    record = {
        "site_id": site_id,
        "pair_id": pair_id,
        "group": group,
        "period": period,
        "start_date": start_date,
        "end_date": end_date,
        "sensor": "Sentinel-2 Level-2A",
        "collection": SENTINEL2_COLLECTION,
        "resolution_m": EXPORT_SCALE_METERS,
        "export_crs": EXPORT_CRS,
        "bands": ",".join(
            BAND_NAMES
        ),
        "scene_count": np.nan,
        "status": None,
        "image_path": str(
            output_file
        ),
        "preview_path": str(
            preview_file
        ),
        "file_size_bytes": np.nan,
        "error": None,
    }

    if (
        output_file.exists()
        and not OVERWRITE_EXISTING
    ):

        record["status"] = "existing"

        record[
            "file_size_bytes"
        ] = output_file.stat().st_size

        if (
            CREATE_RGB_PREVIEWS
            and not preview_file.exists()
        ):

            create_rgb_preview(
                output_file,
                preview_file,
            )

        return record

    try:

        region = shapely_to_ee_geometry(
            site_row.geometry
        )

        image, scene_count = (
            build_sentinel2_composite(
                region=region,
                start_date=start_date,
                end_date=end_date,
            )
        )

        record["scene_count"] = scene_count

        if image is None:

            record["status"] = "no_scenes"

            record["error"] = (
                "No Sentinel-2 scenes were found "
                "for this site and period."
            )

            return record

        output_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        last_error = None

        for attempt in range(
            1,
            RETRY_ATTEMPTS + 1,
        ):

            try:

                print(
                    f"\nDownloading {site_id} {period}; "
                    f"attempt {attempt}..."
                )

                geemap.ee_export_image(
                    image,
                    filename=str(
                        output_file
                    ),
                    scale=EXPORT_SCALE_METERS,
                    region=region,
                    crs=EXPORT_CRS,
                    file_per_band=False,
                )

                if output_file.exists():

                    break

                raise FileNotFoundError(
                    "geemap finished without creating "
                    "the expected Sentinel-2 GeoTIFF."
                )

            except Exception as download_error:

                last_error = download_error

                print(
                    f"Attempt {attempt} failed for "
                    f"{site_id} {period}: "
                    f"{download_error}"
                )

                if attempt < RETRY_ATTEMPTS:

                    time.sleep(
                        RETRY_WAIT_SECONDS
                    )

        if not output_file.exists():

            raise RuntimeError(
                f"Sentinel-2 image was not created after "
                f"{RETRY_ATTEMPTS} attempts. "
                f"Last error: {last_error}"
            )

        record["status"] = "success"

        record[
            "file_size_bytes"
        ] = output_file.stat().st_size

        if CREATE_RGB_PREVIEWS:

            create_rgb_preview(
                output_file,
                preview_file,
            )

        return record

    except Exception as error:

        record["status"] = "failed"
        record["error"] = str(error)

        print(
            f"\nFailed: {site_id}, {group}, {period}"
        )

        print(error)

        return record


# =============================================================================
# 17. Test one treatment site
# =============================================================================

if TEST_DOWNLOAD_FIRST:

    test_site = (
        treatment_sites
        .iloc[0]
    )

    print(
        "\nTesting Sentinel-2 download for:"
    )

    print(
        test_site["site_id"]
    )

    test_record = download_site_image(
        test_site,
        "before",
    )

    print(
        "\nTest result:"
    )

    print(
        test_record
    )

    if test_record["status"] not in {
        "success",
        "existing",
    }:

        raise RuntimeError(
            "The test Sentinel-2 download did not succeed. "
            "Resolve the error before running the complete loop."
        )


# =============================================================================
# 18. Download all Sentinel-2 images
# =============================================================================

download_records = []

total_downloads = (
    len(all_sites) *
    len(IMAGE_PERIODS)
)

progress_bar = tqdm(
    total=total_downloads,
    desc="Downloading Sentinel-2 image chips",
)

for _, site_row in all_sites.iterrows():

    for period in [
        "before",
        "after",
    ]:

        record = download_site_image(
            site_row,
            period,
        )

        download_records.append(
            record
        )

        progress_bar.update(
            1
        )

progress_bar.close()


# =============================================================================
# 19. Save Sentinel-2 image inventory
# =============================================================================

sentinel2_inventory = pd.DataFrame(
    download_records
)

SENTINEL2_INVENTORY_FILE = (
    SENTINEL2_DIR /
    "sentinel2_image_inventory.csv"
)

sentinel2_inventory.to_csv(
    SENTINEL2_INVENTORY_FILE,
    index=False,
)

print(
    "\nSentinel-2 inventory saved to:"
)

print(
    SENTINEL2_INVENTORY_FILE
)


# =============================================================================
# 20. Display download status
# =============================================================================

print(
    "\nDownload status counts:"
)

print(
    sentinel2_inventory[
        "status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nStatus by group and period:"
)

print(
    sentinel2_inventory
    .groupby(
        [
            "group",
            "period",
            "status",
        ]
    )
    .size()
)


# =============================================================================
# 21. Check before-after completeness by site
# =============================================================================

successful_statuses = [
    "success",
    "existing",
]

successful_inventory = (
    sentinel2_inventory
    .loc[
        sentinel2_inventory[
            "status"
        ].isin(
            successful_statuses
        )
    ]
    .copy()
)

site_completeness = (
    successful_inventory
    .pivot_table(
        index=[
            "site_id",
            "pair_id",
            "group",
        ],
        columns="period",
        values="image_path",
        aggfunc="first",
    )
    .reset_index()
)

if "before" not in site_completeness.columns:

    site_completeness["before"] = np.nan

if "after" not in site_completeness.columns:

    site_completeness["after"] = np.nan

site_completeness[
    "has_before"
] = site_completeness[
    "before"
].notna()

site_completeness[
    "has_after"
] = site_completeness[
    "after"
].notna()

site_completeness[
    "complete_before_after"
] = (
    site_completeness[
        "has_before"
    ]
    &
    site_completeness[
        "has_after"
    ]
)

SITE_COMPLETENESS_FILE = (
    SENTINEL2_DIR /
    "sentinel2_site_completeness.csv"
)

site_completeness.to_csv(
    SITE_COMPLETENESS_FILE,
    index=False,
)

print(
    "\nSite before-after completeness:"
)

print(
    site_completeness[
        "complete_before_after"
    ].value_counts()
)


# =============================================================================
# 22. Check complete matched treatment-counterfactual pairs
# =============================================================================

site_period_status = (
    successful_inventory
    .assign(
        available=True
    )
    .pivot_table(
        index="pair_id",
        columns=[
            "group",
            "period",
        ],
        values="available",
        aggfunc="max",
        fill_value=False,
    )
)

required_columns = [
    (
        "treatment",
        "before",
    ),
    (
        "treatment",
        "after",
    ),
    (
        "counterfactual",
        "before",
    ),
    (
        "counterfactual",
        "after",
    ),
]

for column in required_columns:

    if column not in site_period_status.columns:

        site_period_status[
            column
        ] = False

site_period_status[
    "complete_pair"
] = (
    site_period_status[
        required_columns
    ]
    .all(
        axis=1
    )
)

matched_pair_completeness = (
    site_period_status
    .reset_index()
)

PAIR_COMPLETENESS_FILE = (
    SENTINEL2_DIR /
    "sentinel2_matched_pair_completeness.csv"
)

matched_pair_completeness.to_csv(
    PAIR_COMPLETENESS_FILE,
    index=False,
)

print(
    "\nMatched-pair completeness:"
)

print(
    matched_pair_completeness[
        "complete_pair"
    ].value_counts()
)


# =============================================================================
# 23. Validate downloaded GeoTIFFs
# =============================================================================

validation_records = []

for _, record in successful_inventory.iterrows():

    image_file = Path(
        record["image_path"]
    )

    validation_record = {
        "site_id": record["site_id"],
        "pair_id": record["pair_id"],
        "group": record["group"],
        "period": record["period"],
        "image_path": str(image_file),
        "exists": image_file.exists(),
        "readable": False,
        "band_count": np.nan,
        "width": np.nan,
        "height": np.nan,
        "crs": None,
        "resolution_x": np.nan,
        "resolution_y": np.nan,
        "valid_pixel_fraction": np.nan,
        "error": None,
    }

    try:

        with rasterio.open(
            image_file
        ) as source:

            data = source.read(
                masked=True
            )

            validation_record[
                "readable"
            ] = True

            validation_record[
                "band_count"
            ] = source.count

            validation_record[
                "width"
            ] = source.width

            validation_record[
                "height"
            ] = source.height

            validation_record[
                "crs"
            ] = str(
                source.crs
            )

            validation_record[
                "resolution_x"
            ] = source.res[0]

            validation_record[
                "resolution_y"
            ] = source.res[1]

            if data.size > 0:

                valid_mask = (
                    ~np.ma.getmaskarray(
                        data
                    )
                )

                validation_record[
                    "valid_pixel_fraction"
                ] = float(
                    valid_mask.mean()
                )

    except Exception as error:

        validation_record[
            "error"
        ] = str(error)

    validation_records.append(
        validation_record
    )

validation_table = pd.DataFrame(
    validation_records
)

VALIDATION_FILE = (
    SENTINEL2_DIR /
    "sentinel2_image_validation.csv"
)

validation_table.to_csv(
    VALIDATION_FILE,
    index=False,
)

print(
    "\nSentinel-2 validation saved to:"
)

print(
    VALIDATION_FILE
)

print(
    "\nReadable image counts:"
)

print(
    validation_table[
        "readable"
    ].value_counts()
)

print(
    "\nBand counts:"
)

print(
    validation_table[
        "band_count"
    ].value_counts(
        dropna=False
    )
)


# =============================================================================
# 24. Create analysis-ready inventory
# =============================================================================

complete_site_ids = set(
    site_completeness.loc[
        site_completeness[
            "complete_before_after"
        ],
        "site_id",
    ]
)

analysis_ready_inventory = (
    successful_inventory
    .loc[
        successful_inventory[
            "site_id"
        ].isin(
            complete_site_ids
        )
    ]
    .copy()
)

ANALYSIS_READY_INVENTORY_FILE = (
    SENTINEL2_DIR /
    "sentinel2_analysis_ready_inventory.csv"
)

analysis_ready_inventory.to_csv(
    ANALYSIS_READY_INVENTORY_FILE,
    index=False,
)

print(
    "\nAnalysis-ready Sentinel-2 inventory saved to:"
)

print(
    ANALYSIS_READY_INVENTORY_FILE
)


# =============================================================================
# 25. Create output folder summary
# =============================================================================

def count_tiff_files(
    folder,
):
    """
    Count GeoTIFF files directly inside a folder.
    """

    return len(
        list(
            Path(folder).glob(
                "*.tif"
            )
        )
    )


folder_summary = pd.DataFrame(
    [
        {
            "group": "treatment",
            "period": "before",
            "folder": str(
                S2_TREATMENT_BEFORE_DIR
            ),
            "tiff_count": count_tiff_files(
                S2_TREATMENT_BEFORE_DIR
            ),
        },
        {
            "group": "treatment",
            "period": "after",
            "folder": str(
                S2_TREATMENT_AFTER_DIR
            ),
            "tiff_count": count_tiff_files(
                S2_TREATMENT_AFTER_DIR
            ),
        },
        {
            "group": "counterfactual",
            "period": "before",
            "folder": str(
                S2_COUNTERFACTUAL_BEFORE_DIR
            ),
            "tiff_count": count_tiff_files(
                S2_COUNTERFACTUAL_BEFORE_DIR
            ),
        },
        {
            "group": "counterfactual",
            "period": "after",
            "folder": str(
                S2_COUNTERFACTUAL_AFTER_DIR
            ),
            "tiff_count": count_tiff_files(
                S2_COUNTERFACTUAL_AFTER_DIR
            ),
        },
    ]
)

FOLDER_SUMMARY_FILE = (
    SENTINEL2_DIR /
    "sentinel2_folder_summary.csv"
)

folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)

print(
    "\nFinal Sentinel-2 folder summary:"
)

print(
    folder_summary
)


# =============================================================================
# 26. Final validation
# =============================================================================

print(
    "\n"
    + "=" * 78
)

print(
    "SENTINEL-2 IMAGE DOWNLOAD SUMMARY"
)

print(
    "=" * 78
)

print(
    f"Total requested images: "
    f"{len(sentinel2_inventory):,}"
)

print(
    f"Successful or existing images: "
    f"{len(successful_inventory):,}"
)

print(
    f"Failed images: "
    f"{(
        sentinel2_inventory['status']
        == 'failed'
    ).sum():,}"
)

print(
    f"No-scene images: "
    f"{(
        sentinel2_inventory['status']
        == 'no_scenes'
    ).sum():,}"
)

print(
    f"Sites with complete before-after images: "
    f"{site_completeness['complete_before_after'].sum():,}"
)

print(
    f"Complete treatment-counterfactual pairs: "
    f"{matched_pair_completeness['complete_pair'].sum():,}"
)

print(
    "\nSentinel-2 treatment before folder:"
)

print(
    S2_TREATMENT_BEFORE_DIR
)

print(
    "\nSentinel-2 treatment after folder:"
)

print(
    S2_TREATMENT_AFTER_DIR
)

print(
    "\nSentinel-2 counterfactual before folder:"
)

print(
    S2_COUNTERFACTUAL_BEFORE_DIR
)

print(
    "\nSentinel-2 counterfactual after folder:"
)

print(
    S2_COUNTERFACTUAL_AFTER_DIR
)

print(
    "\nNotebook completed."
)

Packages loaded successfully.
Sentinel-2 output directory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2
Required site files found.
Earth Engine initialized using existing credentials.
Image periods:
before 2024-08-15 2024-09-23
after 2024-10-01 2024-10-31
Treatment sites: 26
Counterfactual sites: 260
Total sites: 286

Available Sentinel-2 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']

Testing Sentinel-2 download for:
treatment_0001

Test result:
{'site_id': 'treatment_0001', 'pair_id': 'pair_0001', 'group': 'treatment', 'period': 'before', 'start_date': '2024-08-15', 'end_date': '2024-09-23', 'sensor': 'Sentinel-2 Level-2A', 'collection': 'COPERNICUS/S2_SR_HARMONIZED', 'resolution_m': 10, 'export_crs': 'EPSG:32617', 'bands': 'B2,B3,B4,B8,B11,B12,N


Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2/counterfactual/before/counterfactual_0001_01_before_sentinel2.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2/counterfactual/after/counterfactual_0001_01_after_sentinel2.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2/counterfactual/before/counterfactual_0001_02_before_sentinel2.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2/counterfactual/after/counterfactual_0001_02_after_sentinel2.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel2/counterfactual/before/counterfactua